# Time Series Simulation Study
In this notebook you make a simulation study to investigate the class of ARMA models. You will see that this class of models is 
- easily applicable with the `statsmodel` package,
- can be used for simulation of time series data,
- showing good forecasting abilities.

At the end of this notebook you will have learned:
- how to simulate time series data,
- how to define, estimate and forecast with ARMA and SARIMA models,
- how to decompose a time series into a trend, seasonality and rest,
- how to define Exponential Smoothing estimators and apply them to the data for forecasting,
- how to apply stationarty tests and
- how to apply ACF, PACF and Periodogram plots for model diagnosis.

## Load packages

In [ ]:
import numpy as np
import pandas as pd
import warnings
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import kpss
from statsmodels.tsa.stattools import adfuller
from scipy import signal
%matplotlib inline

# If you want a style choose one
#plt.style.use('Solarize_Light2')
#plt.style.use('tableau-colorblind10')
NF_ORANGE = '#ff5a36'
NF_BLUE = '#163251'

See all matplotlib styles under [matplotlib styles](https://problemsolvingwithpython.com/06-Plotting-with-Matplotlib/06.13-Plot-Styles/)

## Define auxiliary functions

The following functions are defined to simplify time series plotting, time differencing and stationarity testing. We will use them through this notebook.

In [ ]:
def plot_ts(ts = None, ts_add = None, title ='Time Series', legend=['1']):
    """
    Plots one or two time series in a single plot
    
        Args:
        ts: 1d- or 2d-array of time series. Dimension
            must be (n,) or (n,2)
        title: Title for the time plot.
        legend: list of legend names. If empty no legend.
        
        Returns:
        matplotlib plot object
    """
    plt.figure(figsize=(10, 4))
    plt.plot(ts[:,], color=NF_ORANGE)
    plt.grid(True,axis='y')
    plt.title(title)
    if ts_add is not None:
        plt.plot(ts_add, color=NF_BLUE)
    if len(legend) > 0:
        plt.legend(legend)
    plt.show()

def plot_acf_pacf(ts, lags=10, layout='h'):
    """
    Plots the empirical ACF and PACF of a time series process.
    
        Args:
        ts: array of time series
        
        Returns:
        matplotlib subplot with ACF and PACF
    """
    if layout == 'h':
        fig, ax = plt.subplots(1, 2, figsize = (10,3))
    else: 
        fig, ax = plt.subplots(2, 1, figsize = (10,3))
    sm.tsa.graphics.plot_acf(ts,color = NF_ORANGE,lags=lags, ax = ax[0])
    sm.tsa.graphics.plot_pacf(ts,color = NF_ORANGE, lags = lags, ax = ax[1])    


def diff_series(ts, interval=1):
    """
    Differences a time series by a certain lag.
    
        Args:
        ts: array of 1d time series
        
        Returns:
        Differenced time series
    """
    diff = ts[interval:] - ts[:-interval]
    return diff

def kpss_test(ts):
    """
    Performs a KPSS test for the null hypothesis of stationarity.
    
        Args:
        ts: 1d time series
        
        Returns:
        Summary of test statistic and critical values
    """
    print ('Results of KPSS Test:')
    kpsstest = kpss(ts, regression='c', nlags='legacy')
    kpss_output = pd.Series(kpsstest[0:3], index=['Test Statistic','p-value','Lags Used'])
    for key,value in kpsstest[3].items():
        kpss_output['Critical Value (%s)'%key] = value
    print (kpss_output)

def adf_test(ts):
    """
    Performs a Dickey-Fuller test for the null hypothesis of
    non-stationarity.
    
        Args:
        ts: 1-d time series
    
        Returns:
        Printed test statistic and critical values.
    """
    print ('Results of Dickey-Fuller Test:')
    dftest = adfuller(ts, autolag='AIC')
    dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','#Lags Used',
                                             'Number of Observations Used'])
    for key,value in dftest[4].items():
        dfoutput['Critical Value (%s)'%key] = value
    print(dfoutput)

## Autoregressive processes
Remember that autoregressive processes model the time dynamics of a variable $X(\omega)=\{1,\ldots,n\}$ by its own last $p$ past values - we call these values also _lags_. The innovations $\varepsilon_t$ are usually considered as [white noise](https://en.wikipedia.org/wiki/White_noise) (zero mean, uncorrelated over time and a constant variance). 

$X_t=\sum_{j=1}^p\phi_jX_{t-j} + \varepsilon_t$

A specialty of these processes is that any shock $\varepsilon_t$ dies out very slowly - with more than $p$ steps (theoretically infinitely many steps). You can make this clear by recursively unfolding the process of $X_t$ (replacing $X_{t-j}$ by its own equation. 

>__Exercise__: Show that in an AR(1) process shocks by $\varepsilon_t$ die out after infinitely many time steps by replacing $X_{t-1}$ recursively with its own equation.

### Autoregressive process of order $p=1$
We begin wih the most simple AR process, namely an AR(1) process, for which you have derived in the exercise above that its shocks die out very slowly over time. 

In an AR(1) process the dynamics of the variable $X_t$ are described only by its first lagged value. A comfortable property of these processes is that the depedency is very clear (also in plots) as second lag effects do not intervene. 

$X_t=\phi X_{t-1} + \varepsilon_t$

In the following we use the `statsmodels` package and more precisely the `tsa.ArmaProcess` object to simulate an AR(1) process with the parameter $\phi=0.7$ and $n=500$ sample points. In difference to cross-sectional data, time series data has to be sequentially simulated and takes longer. 

_Note_: Do not get confused by the `ar` parameters in the constructor: usually an AR process is denoted as

$
(1-\phi L)X_t=\varepsilon_t
$

where $L$ is the socalled _Lag Operator_ shifting the whole time series backwards, i.e. $LX_t=X_{t-1}$. Therefore AR parameters are denoted by their negatives and include the $1$ for the actual value. We will see that this different for MA processes.

In [ ]:
# Sample a simple AR(1) process.
np.random.seed(42)
y = sm.tsa.ArmaProcess(ar=[1,-0.7]).generate_sample(nsample=500)
plot_ts(y, title='', legend=['AR(1)'])

Here we can see the simple and clear structure of the AR(1) process. By the strong autocorrelation to the first lag the process usually exhibits longer periods of high and then longer periods of low values. It stays up a while and then falls down to stay there for some periods.  

### Autoregressive process of order $p=2$

Autoregressive processes of order two are more complex as their AR(1) counterparts as at each time step there exists an interaction of two linear terms. If now have different signs one term is dampening the other and the influence depends highly on the two values $X_{t-1}$ and $X_{t-2}$ (e.g. when they have different signs in value). 

$X_t=\phi_1 X_{t-1} + \phi_2 X_{t-2} + \varepsilon_t$

In this case we model a process with parameters of different sign.

In [ ]:
# Sample an AR(2) processs with diverging parameters. 
np.random.seed(42)
y = sm.tsa.ArmaProcess(ar=[1, -1.5, 0.6]).generate_sample(nsample=500)
plot_ts(y, title='', legend=['AR(2)'])

This process appears to be even more autocorrelated than the AR(1) process. Note, how this process cycles a lot - it oscillates. And this has a certain reason: if we construct a polynomial of the lag operator $L$ of this model

$
1 - 1.5L + 0.6L^2
$
 
and solve for the roots of this polynomial, these roots are complex and as complex numbers can be represented in trigonometric form, i.e. $z=r(\cos(\varphi)+i\sin(\varphi))$ we see an oscillating behavior of the time series here.

### Estimate AR model with OLS
As AR models contain only lagged observed values we can estimate them by plain OLS. 

In [ ]:
ar_model = sm.OLS(y[2:], np.array([y[1:499],y[:498]]).T, hasconst=False)
# Use a heteroscedasticity and autocorrelation-consistent 
# covariance estimator.
res = ar_model.fit(cov_type='HAC', cov_kwds={'maxlags': 4}, use_t=True)
print(res.summary())

As we can see the OLS estimator gets the true parameters $[1.5,-0.6]$ quite well with strong significance. In this case even the $R^2$ is quite high with over 90% of explained variance. The [Durbin-Watson test statistic](https://en.wikipedia.org/wiki/Durbin%E2%80%93Watson_statistic) is closely above $2$ which points to no autocorrelation at lag 1 (for more diagnosis tests see [here](https://www.statsmodels.org/devel/examples/notebooks/generated/regression_diagnostics.html) or [here](https://www.statsmodels.org/stable/diagnostic.html)). Regrettably, in practice, we often do not have such a simple setting and are confronted with rather mixed processes that include not only observed, but also unobserved variables. In such cases we often have an ARMA process (or the ARMA process approximates well enough an infinite MA or AR process).

>__Exercise__: Try out to simulate and estimate an AR(p) process with more than p=6 lagged values. 

## Autoregressive Moving Average processes
ARMA processes are processes that model the dependent variable's time dynamics by lagged values of the variable itself and lagged innovations:

$X_t=\sum_{i=1}^p\phi_i X_{t-i} + \sum_{j=1}^q\theta_j\varepsilon_{t-j} + \varepsilon_t$

The reason behind ARMA processes is that they are a good approximations to the inifnite order MA or AR processes that might occur and that are the theoretical basis of the [_Wold theorem_](https://en.wikipedia.org/wiki/Wold%27s_theorem).

The ARMA process allows to estimate a way higher-order process by a combination of low-order processes, while the Wold theorem assures that there is an underlying process of inifnite order when the time series $X_t$ is covariance-stationary. 

### A simple ARMA process
The most simple ARMA process is an ARMA(1,1) process. 

$X_t = \phi X_{t-1} + \theta\varepsilon_{t-1} + \varepsilon_t$

In the following we will model the process with parameters $\phi=0.6$ and $\theta=-0.2$. 

_Note_: The parameters of the MA term are as they come in the equation - in contrast to the AR parameters. This is due to the fact that the MA model is usually denoted with the _Lag Operator_, we got to know above as 

$
x_t=(1+\theta L)\varepsilon_t~.
$

In this case we sample 10,000 observations:

In [ ]:
# Sample and plot an ARMA(1,1) process
np.random.seed(42)
y = sm.tsa.ArmaProcess(ar=[1, -.6],ma=[1,-.2]).generate_sample(nsample=10000)
plot_ts(y[:1000], legend=['ARMA(1,1)'], title='')

### Estimate ARMA model with MLE
As the ARMA model contains MA terms, i.e. lagged innovations which are usually non-observable we cannot simply apply OLS for estimation. Instead it needs a conditionally dependent probability model with Maximum Likelihood Estimation and this is the usual approach in time series modeling. An ARMA model is defined by its orders $p$ and $q$ and these are the parameters that get passed to `statsmodels`' `tsa.ARMA()` function as a `list`.

In [ ]:
# Construct an ARMA (1,1) process 
# Estimate the ARMA process without a constant
# (in this case we do know
# there is no constant)
arma_model = sm.tsa.arima.ARIMA(y, order=(1,0,1), trend='n')

# Estimate the ARMA model
res = arma_model.fit()
print(res.summary())

We see that estimating an ARMA model with $p=q=1$ gives us estimations that come close to the original parameters $\phi=0.6$ and $\theta=-0.2$ with high significance (`P|z|=0.000`). The summary of the model estimation from `statsmodels` shows us next to the estimators and their statistical properties also the log-likelihood itself and some information criteria that can help us to evaluate a certain model in relation to a benchmark when doing grid search (e.g. AIC). 

In the lower part of the print out we see the roots of the lag polynomial and we can see that these roots are all outside the unit circle and are not complex (which is possible). Roots outside the unit circle guarantee that the estimated model defines a stationary ARMA process. We speak of unit root processes when the process has roots that lay on the unit circle, i.e. we have actually a random walk. Roots below one define an unstable process (i.e. non-stationary). 

>__Exercise__: Read through the description of the model summary and make yourself clear what the single metrics measure (see also [statsmodels.tsa.arima_model.ARMAResults](https://www.statsmodels.org/0.6.1/generated/statsmodels.tsa.arima_model.ARMAResults.html)).

## Empirical ACF and PACF of an ARMA process
In practice, if we want to determine the number of lags to include into an AR/MA or ARMA model we can take advantage of the autocorrelation function (ACF) and partial autocorrelation function (PACF). Both are function of the lag $k$ between two variables in time $X_t$ and $X_{t-k}$. So a observation $k$ days earlier or $k$ days later. The question is how much such observations influence each other either directly (PACF) or directly and indirectly (ACF). If the ACF cuts off after $q$ lags and the PACF only slowly decays it measn we have probably an MA(q) process producing the data at hand. It is the other way around for an AR(p) process that cuts of in PACF at lag $p$ and decays slowly in the ACF. 

#### Empirical ACF and PACF for AR and MA process
In the following we take a look at simulated ARMA processes and their ACF and PACF functions.

In [ ]:
# Sample an AR(3) process and plot the ACF and PACF
np.random.seed(42)
y = sm.tsa.ArmaProcess(ar=[1, -0.4, 0.7, -0.1]).generate_sample(nsample=10000)
plot_acf_pacf(y);

You see here clearly how the AR(3) process cuts off the PACF function at lag $p=3$ (all other lags exhibit only insignificant PACF values). In contrast, the ACF function oscillates down (the oscillation comes also from the alternating signs in the coefficients). This would point towards an AR(3) process and we know that this indeed an AR(3) process.

Let us consider how this looks for an MA(2) process.

In [ ]:
# Sample an MA(2) process and plot the ACF and PACF
np.random.seed(42)
y = sm.tsa.ArmaProcess(ma=[1, -0.3, 0.3]).generate_sample(nsample=10000)
plot_acf_pacf(y)

Here, we see that the ACF function cuts of at lag $q=2$ whereas the PACF function decays more slowly and still resembles some oscillating movement after a couple of lags. This points directly to an MA(2) model and in this case we do know that it is.

#### Empirical ACF and PACF for a full ARMA process
For a true ARMA process results often do not look that clear - but this is of course also a hint. In the following we sample an ARMA process with 3 AR lags and 2 MA lags. 

In [ ]:
# Plot ACR and PACR
np.random.seed(42)
y = sm.tsa.ArmaProcess(ar=[1, -0.4, 0.7, -0.1], ma=[1, -0.3, 0.3]).generate_sample(nsample=10000)
plot_acf_pacf(y)

#### Estimate ARMA model
Both, the PACF function and the ACF show a slow decay and do not give us any suggestion in regard to the lags to include. Let us start a grid search over some combination of lags to decide which model might fit best in regard to the Akaike Information Criterion (AIC).

_Note_: Minimizing the AIC is asymptotically equivalent to minimizing the one-step-head out-of-sample MSE.

In [ ]:
# Make train-test set split
train = y[:8999]
test = y[9000:9999]

# Optimize models 
from collections import OrderedDict
aic = {}
for i in range(1,5):
    for j in range(1,5):
        try:
            np.random.seed(42)
            model = sm.tsa.arima.ARIMA(train, order= (i,0,j), trend='n', enforce_invertibility=False)    
            res = model.fit()
            aic['({},{})'.format(i,j)] = res.aic
            # We need to make sure that non-stationary errors
            # get caught appropriately
        except ValueError:
            aic['({},{})'.format(i,j)] = np.nan
aic = OrderedDict(sorted(aic.items(), key=lambda x: x[1]))
aic

From what the grid search brought out there are many very similar models and the one with the lowest AIC is the one with 3 AR terms and 2 MA terms. In this case we know that this was indeed the data generating process (DGP). If we use these amount of lags in an estimation we can get the following results. 

In [ ]:
# Estimate the model with the lowest AIC: ARMA(3,2)
arma_model = sm.tsa.arima.ARIMA(train, order=(3,0,2), trend='n')
res = arma_model.fit()
print(res.summary())

Parameter estimates all resemble closely the true parameters in the DGP above and show high significance. In this case we have some complex roots (the only real one is greater than 1 and therefore we do not have a unit root). 

## Trend and seasonality
-------------------
As already mentioned in lecture, trends and seasonalities make a time series process non-stationary as they induce time-dependent distributions, i.e. values of $X_t$ occurring between $t$ and $t+h$ are differently distributed than values between times $t+k$ and $t+k+h$. In such cases the standard theory cannot be applied as stationary processes are assumed in this theory. 

It therefore needs a detrending and removing of seasonal components. In the following we consider (1) the composition of time series by a trend a seasonal component and a time-independent dynamic process. For this reason we sample data from a normal distribution (our white noise) and define two trend functions 

$$
\begin{align}
m_t&=\alpha_1+\alpha_2\cdot t\\
m_t&=\beta_1+\beta_2\cdot t + \beta_3\cdot t^2\\
\end{align}
$$

that we add to the white noise.

In [ ]:
# Define a DGP with a linear and quadratic trend
np.random.seed(42)

# Sample a normal white noise 
y = np.random.normal(0.0, 0.4, 100)

# Construct the time series with linear and quadratic trend
trend = 0.1 + 0.07 * np.arange(0,100)
y_ln = trend + y
y_qd = y_ln + 0.001 * np.arange(0,100)**2

# Plot the series
plot_ts(y_ln, y_qd, title = 'Trends', legend=['Linear', 'Qaudratic'])

### Estimate a quadratic trend
You learned that there are two methods to eliminate the trend in a time series. The first was estimating it and eliminate this estimate from the original time series. The other method is time differencing. In what follows we use the first method to eliminate the trend from the time series. 

We define a white noise process with 300 observations and compose a time series process by a qaudratic trend component with parameters $\alpha_1=0.2$, $\alpha_2=0.01$ and $\alpha_3=0.001$ and let the innovations be white noise normally distributed.

In [ ]:
# Define the data generating process
np.random.seed(42)
error = np.random.normal(0.0, 1.1, 300)
trend = 0.2 + 0.01 * np.arange(1, 301) + 0.001 * np.arange(1, 301)**2
y = trend + error

# Plot the time series
plot_ts(y,title='', legend=['Quadratic Trend'])

How can we estimate this trend? The most obvious approach is to use OLS with polynomial features in $t$:

$
x_t=\alpha_1+\alpha_2 t + \alpha_3 t^2
$

to estimate the coefficients $\alpha_i~,i=1,2,3$. 

In [ ]:
# Estimate a linear regression model 

# Start by constructing a time index variable and its square
X = np.array([np.arange(1, 301),np.arange(1, 301)**2]).T

# Use a constant term in the OLS regression
X = sm.tools.tools.add_constant(X)
model = sm.OLS(y, X)
res = model.fit()
print(res.summary())

Surprisingly, the constant term is highly non-significant and with a farout estimate. However, the coefficients for the time index and its square are near the true values of $[0.01, 0.001]$. To see, if the predictions are good enough we plot the time series together with the estimated function.

In [ ]:
# Plot the original and estimated time series
y_est = res.predict(X)
plot_ts(y, y_est, title='', legend=['Orig.', 'Est.'])

That appears to be a good estimate of the quadratic trend immanent in the time series. 

>__Exercise__: Take the original time series and difference the estimated one. Plot your results. What do you see? 

### Seasonality
Next to trends, seasonality has an influence on the distribution(s) of a time series, as seasonality implies that at certain times the process explores other value ranges than at others. This again would be a non-stationary setting and would make the standard theory for time series modeling to fail. 

By eliminating seasonalities (any cyclical pattern) from the time series data such a non-stationary time process can be transformed to a stationary one. Similar to trend elmination also seasonalities can be removed by two main approaches: (1) modeling the cyclical pattern and removing an estimate of this model from the original time series or (2) differencing the original time series at periodical time indices (e.g. a quarterly cyclical pattern has been finished after 4 time steps if data is quarterly),

$
\Delta(4)x_t=x_t-x_{t-4}~.
$

In the following we model the cyclical pattern by a _Fourier Series_ with a single sinus function and period 60 (e.g. 60 days).

In [ ]:
# Define the DGP
np.random.seed(42)

# Define the white noise process
error = np.random.normal(0.0, 1.1, 300)

# Define the seasonal pattern
# Use Fourier Series with a single sinus pattern
# with period 60.
seasonal = np.sin((2*np.pi/60)*np.arange(1,301))

# Combine the two terms and plot the series
y = seasonal + error
plot_ts(y, title = 'Seasonality',legend=['Cycle d=60'])

The cyclical pattern in the time series can be seen clearly. It is also easy to detect that the period of the cyclical pattern is 60 as after 60 time steps the process has arrived at a similar level as where it was before 60 steps. 

>__Exercise__: Model a cyclical pattern by an OLS equation using sinusoidal function of the time. Estimate this model and plot it together with the original time series process seen above. If the estimate is good enough in your eyes, difference this estimate from the original time series and plot the remaining term. 

#### Trend and seasonal component
What we usually observe or encounter in time series is a mixture of both, i.e. some trend and some cyclical pattern (sometimes not even easily detectable). In the following we generate a time series with such properties and you will decompose the model into its different components, i.e. (1) trend, (2) seasonal, and (3) residuals. Residuals might show no white noise properties (e.g. autocorrelation) and that would then need to be modeled as well (e.g. with an ARMA model). 

In the following you create a time series with a normally distributed innovation vector together with a sinuoidal seasonal pattern and a linear trend. 

In [ ]:
# Generate the data with trend and seasonal component
np.random.seed(42)

# Define the white noise process
error = np.random.normal(0.0, 1.1, 300)

# Define the seasonal term
seasonal = np.sin((2*np.pi/60)*np.arange(1,301))

# Define the trend component
trend = 0.01 + 0.05 * np.arange(1, 301) 

# Add all components together and plot them
y = trend + seasonal + error
plot_ts(y, title='Time series with trend and seasonality', legend=[])

This looks already like time series we might observe in the news or in business. As the decomposition is part of time series EDA, there exist pre-defined functions to decompose time series into their three components. In the `statsmodels` module the function `tsa.seasonal_decompose()` can be used to make such a decomposition. For the cyclical component you have to define a period length. (Take also a look at the multiplicative version of it). 

In [ ]:
# Decompose the time series into trend, seasonal component and residuals
y_dec = sm.tsa.seasonal_decompose(y, model = 'additive', period=60)
y_dec.plot()
plt.show();

The residuals show white noise behavior. And to see, if residuals have really no autocorrelations, we could do some diagnostic checks and tests. 

>__Exercise__: For the residuals process from the decomposition in `y_dec`. Plot the ACF and PACF. Take also a look at the [_Ljung-Box-Test_](https://www.statsmodels.org/stable/generated/statsmodels.stats.diagnostic.acorr_ljungbox.html) and apply this test to your residuals.

#### Example with a composed model

Remember: any stationary zero-mean time series can be expressed as a harmonic sum of (infinitely many) sinus and cosinus waves (very seldom this sum is finite - therefore modeling with such a harmonic mean a whole time series is avoided). The periodicity $d$ of a seasonal trend then determines the rate at which a cycle is finished in one time step:

$\frac{2\pi}{d}t$

We can also say $\frac{1}{d}$ is the frequency of the cycle, i.e. if $d=4$ a quarter of the whole cycle is concluded at a single time step. Trigonometrically, the cycle concludes $0.25\cdot2\pi$ _radiants_ per time step and a whole cycle after 4 time steps.

In the following simulation we model the seasonal component as a Fourier Series with a single sinus function and a periodicity of 4. A linear trend is added ogether with an AR(2) process. Note that in this case the white noise is generated inside of the `generate_sample()` method. The innvoation term $\varepsilon$ therewith is white noise, however, the residuum of a time series decomposition will be not, as this process is then an AR(2) process such that residuals from the time decomposiion would be autocorrelated (the Ljung-Box test can be applied here).

In [ ]:
# Generate an AR(2) process
np.random.seed(42)
ar_proc = sm.tsa.ArmaProcess(ar = [1, -.7, -.2]).generate_sample(nsample=1000)

# Define trend and seasonal component
seasonal = 1.3 * np.sin(2*np.pi/4*np.arange(1,1001))
trend = 0.01 + 0.02 * np.arange(1, 1001) 
y = trend + ar_proc + seasonal
plot_ts(y,legend=[],title='Composed time series')

You can see that the cycle is somehow still apparent, even though it got washed out a little by the AR(2) component. 

In a real-world time series approach the first step after looking at the graph above, would be a trend elimination. This time we use the second method for trend removal, i.e. the time differencing approach: 

$
\Delta x_t=x_t - x_{t-1}~.
$

Time differencing works well, if the trend is linear. With a non-linear trend we usually have to difference multiple times to arrive at a stationary process. 

In [ ]:
# Detrend the series
y_diff = diff_series(y)
plot_ts(y_diff, title = '', legend=['1st Diff.'])

First-differencing has worked here well - as we know the true trend is linear, we also know why this differencing has worked so well. 

>__Exercise__: For the model above, use a non-linear trend and repeat the steps. See, if a single differencing step is sufficient to eliminate the trend from the time series. 

What we see quite well in the graph above is a cyclical pattern (to get a better view at it you can reduce the number of samples plotted). At this point our time series is not yet stationary and we have to also eliminate the seasonality from it. Quite often we are confronted with a situation where neither the domain knowledge nor the visualization can clearly tell us what cycles are existent in a time series. To get an insight on the different seasonal patterns and their influence on the whole time series a _periodogram_ is used. The periodogram shows the cycle frequencies dominating the series. Usually, we observe in real-world data many cycles of different frequencies, many of which do not play a role in the modeling. 

The periodogram is concerned with the area of spectral analysis where a time series is viewed as a sum of cosine waves with varying amplitudes (height of a cycle) and frequencies (cycle completion per time step). One goal of spectral analysis is to identify the important frequencies among all of them. The periodogram as one of the tools of spectral analysis tries to graph the relative frequencies of the underlying sum of cosine functions. 

If $A$ is the amplitude, $\frac{1}{d}=\omega$ is the frequency, and $\kappa$ is the phase (the starting point of the wave - a cosine wave has a natural starting point and to change that we need the phase) of a cosine wave, than the cosine wave can be denoted as

$
x_t = A\cos(2\pi\omega t + \kappa)~.
$

>__Exercise__: Program the cosine wave in the formula above and choose a range of 0 to 499 for $t$. Plot the values in a line plot and then change the parameters amplitude, frequency and phase and replot. As a starting example you could use $x_t=2\cos\left(2\pi\frac{1}{50}t + 0.6\pi\right)$

The Periodogram tries to filter out dominant frequencies and therefore evaluates different cosine and sine waves, i.e. cosine and sine waves with different frequency, amplitude and phase parameters. It does so by evaluating the cosine and sine waves along a discrete frequency axis of values $\omega_j=\frac{j}{n},~j=1,2,\ldots,\frac{n}{2}$ named harmonic frequencies (here, $n$ denotes the sample size) and considers these frequencies the features of a linear regression: 

$
x_t = \sum_{j=1}^{\frac{n}{2}}\left(\beta_1\left(\frac{j}{n}\right)\cos(2\pi\omega_jt)+\beta_2\left(\frac{j}{n}\right)\sin(2\pi\omega_jt)\right) + \varepsilon_t~,
$

with regression parameters $\beta_1\left(\frac{j}{n}\right)$ and $\beta_2\left(\frac{j}{n}\right)$. It is actually not necessary to carry out this regression, instead a procedure called [_Fast Fourier Transform (FFT)_](https://en.wikipedia.org/wiki/Fast_Fourier_transform) is used that can calculate the parameters in this case very quickly. After regression parameters have been calculated the following sum of squared regression parameters at frequency $\omega_j$ are defined,

$
P\left(\frac{j}{n}\right)=\hat{\beta}_1^2\left(\frac{j}{n}\right) + \hat{\beta}_2^2\left(\frac{j}{n}\right)~,
$

and this is the Periodogram value at the frequency $\omega_j=\frac{j}{n}$. So the regression show which parameters are larger in absolute value and therefore dominate the linear regression. The residual $\varepsilon_t$ not explained by the waves might be still autocorrelated via an ARMA process or some non-linear dynamics. 

In [ ]:
# We use for the sampling frequency 1 as we want to
# discover cycles over the original time steps.
f, Pxx = signal.periodogram(y_diff, fs = 1, window='hamming', scaling='spectrum')
plt.plot(f, Pxx, color = NF_ORANGE)
plt.title('Periodogram')
plt.show();

The dominant frequency here identified by the periodogram lays at around $\frac{1}{d}=0.25$ which corresponds to a periodicity of $d=4$, exactly the periodicity of the simulated seasonality. We can also observe that there exist many more cyclical patterns with other frequencies (observe the little trembling along the line), however the one at frequency 0.25 is the by far most dominant one. By using the periodogram we can identify what frequencies are included in our time series and which are the most dominant ones. By eliminating the most dominant ones we arrive at some point at a stationary time series that allows us explicit modeling via an ARMA model for example. 

As the periodicity in our time series appears to be at 4 (identified by the periodogram), we clean our series from this cyclical pattern by taking a difference between the actual value and its lag four time periods before:

$
\Delta(4) x_t=x_t-x_{t-4}~.
$

In [ ]:
# Difference the time series and plot
y_diff_4 = diff_series(y_diff, 4)
plot_ts(y_diff_4[:200,], title='', legend=['Diff(1,4)'])

From the graph it appears that the dominant cycle which was clearly visual has been removed from the series. If this is the case should be checked by a periodogram on the cleaned time series and also a stationarity test. We look at such stationarity tests in the next section. 

#### Testing for unit roots (non-stationarity)
After differencing to eliminate trend and seasonal components, we want to check, if we were able to arrive at a stationary time series. There exist some main procedures for testing on stationarity, namely [Philipps Perron](https://en.wikipedia.org/wiki/Phillips%E2%80%93Perron_test#cite_note-2) test, [augmented Dickey-Fuller (ADF)](https://en.wikipedia.org/wiki/Augmented_Dickey%E2%80%93Fuller_test) test, and the [Kwiatkowski-Phillips-Schmidt-Shin (KPSS)](https://en.wikipedia.org/wiki/KPSS_test). 

In contrast to the Phillips-Perron and augmented Dickey-Fuller the KPSS test the other way around, i.e. it tests on (trend-)stationarity, where trend-stationarity is the property of a time series to be stationary after elimination of a linear trend (by e.g. first-differencing it). The other two tests test the null of non-stationarity (unit-root).

Here we only apply the augmented Dickey-Fuller and KPSS tests to the data.

In [ ]:
# Performa an augmented Dickey-Fuller test on non-stationarity.
adf_test(y_diff_4)

As the $p$-value is clearly below 1% we can reject the null hypothesis of non-stationarity. 

To be sure we make also an KPSS test that tests the other way around: a null hypothesis of (trend-)stationarity. 

In [ ]:
# Perform KPSS test on stationarity
kpss_test(y_diff_4)

The p-value is 10% and we cannot reject the null hypothesis of stationarity for our differenced time series. 

In regard to the tests we have with high certainty a time series that is indeed stationary. With stationary time series we can now proceed our time series analysis and estimate an ARMA model. 

>__Exercise__: Apply also the Phillips-Perron test and write a corresponding test function as in `kpss_test()` or `adf_test()` to report results. 

### Estimate SARIMA model
As finally we want at to predict the original values (i.e. $x_t$ and not any $\Delta x_t$) we use a socalled SARIMA model that does the differencing for us internally (this is the **I**ntegrating part of the SARIMA). From our EDA above, we found out that a first-order difference eliminates the linear trend and a fourth-order difference eliminates the seasonality in our data. 

__Note__: The order in which we difference does not matter. If we first eliminate the trend and then the seasonality or the other way around produces the same series. 

For defining the lags for the SARIMA model (we do have the integrating lags for the process, namely 1 and 4) we take a look at the ACF and PACF plots.  warnings.warn(


In [ ]:
# Consider the ACF and PACF to detect autocorrelation
# patterns
plot_acf_pacf(y_diff_4)

The plots are still ambiguous in what lag polynomial exists in the SARIMA model. It looks like the ACF cuts off at around the 4th or 5th lag, while the PACF appears to oscillate (which could point towards a not yet detected cycle in the data (the stationarity tests say with great certainty that such cycles do not exist). This might point us to a MA(5) process. 

>__Exercise__: Apply a periodogram on the cleaned data `y_diff_4` and investigate the frequencies the periodogram identifies. Make your own conclusions. 

As the PACF and ACF plots are inconclusive we decide for a grid search on a couple of MA and AR lag combinations to find the SARIMA model that fits the data best. As an evaluation metric we choose the Akaike Information Criterion (we use this criterion already in semi-supervised learning - see for an intuitive explanation the comment below the first model summary) 

In [ ]:
# As we are unsure how many lags to include, 
# we run a grid search and check for the Akaike 
# Criterion to choose a best model. 

# Optimize models 
from collections import OrderedDict
aic = {}
for i in range(0,3):
    for j in range(0,3):
        try:
            # As the MLE optimizer uses Gradient descent we use 
            # a random seed.
            np.random.seed(42)
            # We use a SARIMA model with an integrated order of 1,
            # i.e. first difference eliminates trend, and a
            # integrated seasonal order of 4, i.e. fourth-order
            # differencing eliminates cycle.
            model = sm.tsa.SARIMAX(y, order = (i, 1, j), seasonal_order=(i, 1, j, 4), enforce_invertibility=False, enforce_stationarity=False)    
            res = model.fit(max_iter=2000)
            # We store the Akaike Information Criterion
            aic['({},{})'.format(i,j)] = res.aic
        # We need to make sure that non-stationary errors
        # get caught appropriately
        except ValueError:
            aic['({},{})'.format(i,j)] = np.nan
aic = OrderedDict(sorted(aic.items(), key=lambda x: x[1]))
aic

The grid search suggests a model with 2 MA lags and 2 AR lags - although an MA(2) model comes very near in regard to AIC. From the grid search we decide for an ARMA(2,2) model (note: the ARMA model is an approximation in polynomial anyway - the Wold theorem ensures that a model of linear combinations of errors exists and the ARMA approximates the infinite order polynomial with finite orders of lags in both MA and AR terms). 

#### Final Model 
The grid search points towards a model with 2 AR terms and 2 MA terms, and we estimate this model and use it then for in-sample forecasting.

In [ ]:
# The grid search suggests a SARIMA model with 2 AR lag and 2 MA lags
# for both the main component and the seasonal component.
np.random.seed(42)
sarima_model = sm.tsa.SARIMAX(y, order = (2, 1, 2), seasonal_order=(2, 1, 2, 4), enforce_invertibility=False, enforce_stationarity=False)
res = sarima_model.fit()
print(res.summary())

It might surprise that the model estimation results in almost all parameters being (highly) insignificant, however, a SARIMA model of this form is rather complex and needs usually more data to converge to the true parameters. 

#### Forecasting with SARIMA
With the final estimated model we can do forecasting by using the models `predict()` function. We predict the last 20 values in our sample as a toy example for forecasting and calculate our in-sample forecasting RMSE. 

In [ ]:
# Let us make some predictions for the last indices 
# in the time series
pred = res.predict(980, 999)
y_pred = np.ndarray((100,))

# Set the points not forecasted to NaN 
# (this does not plot them)
y_pred[:80] = np.NaN 
y_pred[80:] = pred

# Print the forecast RMSE.
print('In-sample RMSE: {}'.format(np.sqrt(np.sum((y[980:] - pred)**2))))
plot_ts(y[900:], y_pred, title = '', legend=['Orig.', 'Est.'])

Of course this does not look to bad - this is in-sample forecasted. But we can see from this example how the SARIMA model is able to model the time series with all of its movements quite well. 

## Exponential Smoothing
As mentioned in lecture a very strong benchmark class of time series models is the class of exponential smoothing models, where the actual values are modeled as exponential averages between the last value and past values:

$
\hat{x}_{t+1}=\alpha x_t + (1-\alpha)\hat{x}_{t-1}~.
$

These exponential smoothers are simple, but hard to beat. 

### Import the exponential smoothing functions
We will use a very simple exponential smoothing method and then the well-known [Holt-Winters](https://otexts.com/fpp2/holt-winters.html) smoother.

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.api import SimpleExpSmoothing

### Simple exponential smoothing without seasonality and trend
We use in the following the time series process simulated above for the SARIMA model. The simple exponential smoothing method takes only an average over past values and does assume no trend or seasonality in the data. We should expect that it does not perform well. 

In [ ]:
# Use a smoothing level alpha of 0.4, i.e.
# past values will be given a weight of 60%
# actual values a weight of the remaining 40% 
simple_exp = SimpleExpSmoothing(y, initialization_method='estimated').fit(smoothing_level=0.6,optimized=False)

# Predict the last 20 values in the original
# time series
pred = simple_exp.predict(980, 999)
y_pred = np.ndarray((100,))

# Set the points not forecasted to NaN 
# (this does not plot them)
y_pred[:80] = np.NaN 
y_pred[80:] = pred

# Print the forecast RMSE.
print('In-sample RMSE: {}'.format(np.sqrt(np.sum((y[980:] - pred)**2))))
plot_ts(y[900:], y_pred, title = '', legend=['Orig.', 'Est.'])

As expected the simple exponential smoother that does not take into account the linear trend apparent in the simulated time series data does bad. We have an RMSE of around 7.5 (whereas the RMSE for the SARIMA model was at 4.2. We see that the exponential smoother exhibits some forerunning, which is caused by neglecting two string dynamics in the simulated time series. 

### Holt-Winters exponential smoothing with trend and seasonality
Holt-Winters is a smoothing approach that is usually very hard to beat in practice. Let us see what the smoother can achieve on this simulated data. In contrast to the simple exponential smoother, the Holt-Winters method accounts for a trend and a seasonal component in the data. We can choose either an _additive_ or _multiplicative_ approach. As we do know that the trend and seasonal component have been added and not multiplicated to the white noise we choose the additive method `add` for these two components of the Holt-Winters smoother. 

In [ ]:
# Use an additive seasonality and additive trend. 
# As we know from prior EDA, we have a cycle of 4. 
holt_winters = ExponentialSmoothing(y, seasonal_periods=4,seasonal='add', trend='add', initialization_method='estimated')
res = holt_winters.fit()

# Make a prediction for the last 20 values. 
pred = res.predict(980, 999)
y_pred = np.ndarray((100,))

# Set the points not forecasted to NaN 
# (this does not plot them)
y_pred[:80] = np.NaN 
y_pred[80:] = pred

# Print the forecast RMSE.
print('In-sample RMSE: {}'.format(np.sqrt(np.sum((y[980:1000] - pred)**2))))
plot_ts(y[900:], y_pred, title = '', legend=['Orig.', 'Est.'])

As expected the Holt-Winters exponential smoothing performs almost a sgood as our SARIMA method on in-sample prediction. It is a strong benchmark model to use in time series analysis. 

Even though we know how the data was generated and used a SARIMA model (with slightly different coefficients than the simulated process), Holt-Winters has only a slightly greater RMSE on the in-sample forecast using a way less computative approach. 

>__Exercise__: Perform the forecasting by using a train-test split and repeat the steps above for the comnposed time series process with trend and seasonality. Feel free to simulate more data for your estimation approach. 

>__Exercise__:Simulate your own process by choosing the number of lags and parameters respectively and perform all steps from above for your simulated process. Compare your results with other groups. 